# 🟠 Day 7 — Memory with Checkpointers

**LangGraph from Zero to Advanced · Day 7 of 16**

**Today's goal:** make your chatbot and agents **remember** conversations automatically — no more passing history by hand.

| Lesson | Topic |
|---|---|
| 1 | Checkpointer + `thread_id` = memory |
| 2 | Separate conversations with separate threads |
| 3 | Look inside: `get_state` |
| 4 | Every step is saved: `get_state_history` |
| 5 | Memory that survives a restart (SQLite) |
| 6 | Memory for agents |
| 7 | Summarize long chats |
| 8 | Practice |

In [ ]:
%pip install -q -U langgraph langchain langchain-openai langgraph-checkpoint-sqlite python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — Checkpointer + `thread_id` = memory

A **checkpointer** saves the graph's state after every step. A **`thread_id`** is the name of a conversation.

* Compile with a checkpointer: `graph.compile(checkpointer=InMemorySaver())`
* Pass the thread in the config: `config = {"configurable": {"thread_id": "chat-1"}}`
* Next time you invoke with the **same** thread, LangGraph loads the saved state first — and `add_messages` appends your new message to it.

```
invoke #1 (thread "chat-1") ──► state saved
invoke #2 (thread "chat-1") ──► saved state loaded ──► new message appended ──► saved again
```

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage

SYSTEM = SystemMessage(content="You are a friendly assistant. Keep answers short.")

def chatbot(state: MessagesState) -> dict:
    return {"messages": [llm.invoke([SYSTEM] + state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

memory = InMemorySaver()
chat_app = builder.compile(checkpointer=memory)    # 👈 the only change

In [ ]:
config = {"configurable": {"thread_id": "chat-1"}}

r1 = chat_app.invoke({"messages": [("user", "Hi! I'm Priya and I work as a data scientist.")]}, config)
print("🤖", r1["messages"][-1].content)

# Only the NEW message is sent — no history passed!
r2 = chat_app.invoke({"messages": [("user", "What's my name and job?")]}, config)
print("🤖", r2["messages"][-1].content)

print("\nMessages saved in chat-1:", len(r2["messages"]))
assert len(r2["messages"]) == 4
print("✅ The bot remembers!")

## Lesson 2 — Separate conversations

Each `thread_id` is a separate memory — like separate WhatsApp chats. This is how one app serves many users.

In [ ]:
other = {"configurable": {"thread_id": "chat-2"}}
r = chat_app.invoke({"messages": [("user", "What's my name?")]}, other)
print("chat-2 🤖", r["messages"][-1].content)       # doesn't know Priya
print("chat-2 messages:", len(r["messages"]))
assert len(r["messages"]) == 2

## Lesson 3 — Look inside: `get_state`

`get_state(config)` returns a **snapshot** of a thread:

| Field | Meaning |
|---|---|
| `.values` | the current state |
| `.next` | which node(s) run next (empty = finished) |
| `.config` | ids of this checkpoint |
| `.metadata` | step number, which node wrote it, … |

In [ ]:
snapshot = chat_app.get_state(config)
print("next      :", snapshot.next)
print("step      :", snapshot.metadata["step"])
print("checkpoint:", snapshot.config["configurable"]["checkpoint_id"])
for m in snapshot.values["messages"]:
    print(f"  {m.type:5} | {m.content[:60]}")

## Lesson 4 — Every step is saved: `get_state_history`

The checkpointer keeps **every** checkpoint, newest first. This powers debugging, "undo" and time travel (Day 8).

In [ ]:
for snap in chat_app.get_state_history(config):
    print(f"step {snap.metadata['step']:>2} | messages={len(snap.values.get('messages', []))} | next={snap.next}")

## Lesson 5 — Memory that survives a restart (SQLite)

`InMemorySaver` forgets everything when the notebook restarts. `SqliteSaver` writes to a file.

| Checkpointer | Package | Use for |
|---|---|---|
| `InMemorySaver` | `langgraph` | learning, tests |
| `SqliteSaver` | `langgraph-checkpoint-sqlite` | local apps, demos |
| `PostgresSaver` | `langgraph-checkpoint-postgres` | production |

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("day07_memory.db", check_same_thread=False)
sqlite_memory = SqliteSaver(conn)
persistent_app = builder.compile(checkpointer=sqlite_memory)

cfg = {"configurable": {"thread_id": "priya-forever"}}
persistent_app.invoke({"messages": [("user", "Remember: my favourite food is pav bhaji.")]}, cfg)
print("Saved to day07_memory.db ✅")

🧪 **Try it:** restart the kernel (*Kernel → Restart*), run the setup cell + the cell below. The bot still remembers!

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

reloaded_app = builder.compile(checkpointer=SqliteSaver(sqlite3.connect("day07_memory.db", check_same_thread=False)))
r = reloaded_app.invoke({"messages": [("user", "What's my favourite food?")]},
                        {"configurable": {"thread_id": "priya-forever"}})
print("🤖", r["messages"][-1].content)

## Lesson 6 — Memory for agents

Any graph can get memory — including the tool agents from Day 6. With `create_agent`, pass `checkpointer=`.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def add_expense(item: str, amount: float) -> str:
    """Record an expense (amount in rupees)."""
    return f"Recorded ₹{amount} for {item}"

expense_agent = create_agent(
    model=llm,
    tools=[add_expense],
    system_prompt="You track expenses. Use the tool to record each expense. You can add totals from the conversation.",
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "expenses-oct"}}
for text in ["I spent 250 on lunch", "And 120 on an auto ride", "What's my total so far?"]:
    out = expense_agent.invoke({"messages": [("user", text)]}, cfg)
    print("👤", text, "\n🤖", out["messages"][-1].content, "\n")

## Lesson 7 — Summarize long chats

With memory, history grows forever → slow and expensive. A common fix:

1. When there are more than **6** messages, a `summarize` node writes a short **summary**.
2. It **deletes** old messages with `RemoveMessage`, keeping only the last 2.
3. The chatbot adds the summary to its system prompt.

```
START ► chatbot ─(more than 6 messages?)─yes─► summarize ► END
                           └──────────no──────────────────► END
```

In [ ]:
from typing import Literal
from langchain_core.messages import RemoveMessage, HumanMessage

class SummaryState(MessagesState):
    summary: str

def chat_with_summary(state: SummaryState) -> dict:
    system = "You are a friendly assistant. Keep answers short."
    if state.get("summary"):
        system += f"\nSummary of the earlier conversation: {state['summary']}"
    return {"messages": [llm.invoke([SystemMessage(content=system)] + state["messages"])]}

def summarize(state: SummaryState) -> dict:
    prompt = (f"Current summary: {state.get('summary', '(none)')}\n"
              "Extend the summary with the new messages above. Max 3 sentences.")
    summary = llm.invoke(state["messages"] + [HumanMessage(content=prompt)]).content
    delete = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": summary, "messages": delete}

def need_summary(state: SummaryState) -> Literal["summarize", "__end__"]:
    return "summarize" if len(state["messages"]) > 6 else END

g = StateGraph(SummaryState)
g.add_node("chatbot", chat_with_summary)
g.add_node("summarize", summarize)
g.add_edge(START, "chatbot")
g.add_conditional_edges("chatbot", need_summary)
g.add_edge("summarize", END)
summary_app = g.compile(checkpointer=InMemorySaver())
show_graph(summary_app)

In [ ]:
cfg = {"configurable": {"thread_id": "long-chat"}}
for text in ["I'm Arjun from Nagpur.", "I'm learning LangGraph.", "I like chess.", "My dog is called Bruno."]:
    summary_app.invoke({"messages": [("user", text)]}, cfg)

state = summary_app.get_state(cfg).values
print("Messages kept:", len(state["messages"]))
print("Summary:", state.get("summary"))

r = summary_app.invoke({"messages": [("user", "What's my dog's name and where am I from?")]}, cfg)
print("\n🤖", r["messages"][-1].content)

## Lesson 8 — Practice

**Exercise 1.** Use one `chat_app` with two threads, `"user-a"` and `"user-b"`. Tell each a different favourite colour, then ask both. Check they don't mix.

**Exercise 2.** Write a function `count_turns(app, thread_id)` that returns how many **human** messages are saved in a thread.

In [ ]:
# ✅ Solution 1
app = builder.compile(checkpointer=InMemorySaver())
a = {"configurable": {"thread_id": "user-a"}}
b = {"configurable": {"thread_id": "user-b"}}
app.invoke({"messages": [("user", "My favourite colour is blue.")]}, a)
app.invoke({"messages": [("user", "My favourite colour is green.")]}, b)
print("A:", app.invoke({"messages": [("user", "My favourite colour?")]}, a)["messages"][-1].content)
print("B:", app.invoke({"messages": [("user", "My favourite colour?")]}, b)["messages"][-1].content)

In [ ]:
# ✅ Solution 2
def count_turns(app, thread_id: str) -> int:
    values = app.get_state({"configurable": {"thread_id": thread_id}}).values
    return sum(1 for m in values.get("messages", []) if m.type == "human")

print("user-a turns:", count_turns(app, "user-a"))
assert count_turns(app, "user-a") == 2
assert count_turns(app, "nobody") == 0
print("✅ All exercises solved!")

## 🎯 Day 7 Recap

* `compile(checkpointer=...)` + `{"configurable": {"thread_id": ...}}` = automatic memory.
* Each thread is a separate conversation.
* `get_state` shows the current snapshot; `get_state_history` shows every saved step.
* `InMemorySaver` for learning, `SqliteSaver`/`PostgresSaver` for real apps.
* Summarize + `RemoveMessage` to keep long chats small.

**Tomorrow (Day 8):** pause the graph and ask a **human** for approval — human-in-the-loop & time travel.